# sharp-steer-vla: steering-Hessian sharp directions vs closed-loop CAA steering of SmolVLA on LIBERO-Goal
**Branch:** `feature/sharp-steer-vla` | **Gate:** pre-registered in README.md (frozen 2026-11-08)
**One-Click Execution:** Runtime -> Run all (L4). Clones, installs lerobot + LIBERO, runs a smoke campaign (includes the GPU
pytest pre-flight; aborts if invalid), then the resumable campaign with results on Google Drive, prints RESULTS.md,
unassigns the GPU. Re-running after a disconnect resumes from Drive. Credit cap: `SSV_MAX_HOURS` (default 3.3 h).

In [ ]:
# Step 1: Credit Guard (auto-unassign on any error or completion) + Colab Secrets
import os, sys, signal, subprocess
from IPython import get_ipython

def auto_teardown(signum=None, frame=None):
    print("\n[Credit Guard] Error, interrupt, or completion! Flushing Drive and unassigning GPU...")
    try:
        from google.colab import drive
        drive.flush_and_unmount()
    except Exception:
        pass
    try:
        from google.colab import runtime
        runtime.unassign()
    except Exception:
        pass

def custom_exc_handler(self, etype, value, tb, tb_offset=None):
    self.showtraceback((etype, value, tb), tb_offset=tb_offset)
    auto_teardown()
    return []

ip = get_ipython()
if ip is not None:
    ip.set_custom_exc((Exception,), custom_exc_handler)

try:
    from google.colab import userdata
    for k in ["HF_TOKEN", "HUGGING_FACE_HUB_TOKEN", "hf_token"]:
        try:
            tok = userdata.get(k)
            if tok:
                os.environ["HF_TOKEN"] = tok
                os.environ["HUGGING_FACE_HUB_TOKEN"] = tok
                print(f"[Secrets] Loaded Hugging Face token from Colab Secrets ({k}).")
                break
        except Exception:
            pass
except ImportError:
    pass

print("[Ready] Credit guard active. VM will self-terminate on error or completion.")

In [ ]:
# Step 2: GPU check + clone repository
import torch
assert torch.cuda.is_available(), "Switch Colab runtime to GPU (L4): Runtime -> Change runtime type -> GPU (L4)"
print(f"GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB | torch {torch.__version__} | CPUs {os.cpu_count()} | Python {sys.version.split()[0]}")
assert sys.version_info >= (3, 12), "lerobot 0.7 needs Python >= 3.12"

repo_url = "https://github.com/jaimejarauta/decoupled-delta-net.git"
branch = "feature/sharp-steer-vla"
target_dir = "/content/sharp-steer-vla"
if not os.path.exists(target_dir):
    subprocess.run(["git", "clone", "-b", branch, repo_url, target_dir], check=True)
else:
    subprocess.run(["git", "-C", target_dir, "fetch", "origin", branch], check=True)
    subprocess.run(["git", "-C", target_dir, "reset", "--hard", f"origin/{branch}"], check=True)
os.chdir(target_dir)
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

In [ ]:
# Step 3: Installs (lerobot 0.7.0 with SmolVLA + LIBERO extras), headless MuJoCo rendering, LIBERO config
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "cmake"], check=False)  # hf-egl-probe builds with CMake (no py3.13 wheel)
_r = subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], capture_output=True, text=True)
if _r.returncode:  # Colab hides child-process output: print pip's real error before failing
    print(_r.stdout[-4000:]); print(_r.stderr[-6000:])
_r.check_returncode()
# LIBERO asks for its data paths on first import; answer "N" (defaults) once
print(subprocess.run([sys.executable, "-c", "from libero.libero import benchmark; print('LIBERO ok', benchmark.get_benchmark_dict()['libero_goal']().n_tasks, 'goal tasks')"],
                     input="N\n", capture_output=True, text=True).stdout[-300:])
if os.environ.get("HF_TOKEN"):
    import huggingface_hub
    huggingface_hub.login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)
print(subprocess.run([sys.executable, "-c", "import lerobot, transformers, torch; print('lerobot', lerobot.__version__, '| transformers', transformers.__version__, '| torch', torch.__version__)"],
                     capture_output=True, text=True, check=True).stdout)

In [ ]:
# Step 4: Mount Drive; results (incl. resume state) and the HF cache live there
from google.colab import drive
drive.mount("/content/drive")
os.environ["SSV_RESULTS"] = "/content/drive/MyDrive/moonshot_results/sharp-steer-vla"
os.environ["HF_HOME"] = "/content/hf_cache"
os.makedirs(os.environ["SSV_RESULTS"], exist_ok=True)
print("Results dir:", os.environ["SSV_RESULTS"], "| existing:", sorted(os.listdir(os.environ["SSV_RESULTS"])))

In [ ]:
# Step 5: Pre-flight = smoke campaign in a throw-away dir. It runs the GPU pytest first (zero-vector hook is a no-op,
# steering reaches the actions, gradient finite, FD-Hessian symmetric, postprocessor affine, LIBERO envs step) and
# ABORTS (and unassigns) on failure; then the whole pipeline with 2 episodes x 20 steps, so a crash costs minutes, not CU.
import shutil
smoke_dir = os.path.join(os.environ["SSV_RESULTS"], "smoke")
shutil.rmtree(smoke_dir, ignore_errors=True)
r = subprocess.run([sys.executable, "-u", "run_all.py", "--smoke"], env=dict(os.environ, SSV_RESULTS=smoke_dir))
if r.returncode:
    raise RuntimeError("smoke / pytest pre-flight failed -> campaign aborted (see output above)")
print("[Pre-flight] pytest + smoke pipeline passed on GPU.")

In [ ]:
# Step 6: Full campaign (resumable): pytest -> states -> pilot -> Hessians -> main cells -> gate
p = subprocess.Popen([sys.executable, "-u", "run_all.py"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    if "Loading weights" not in line:
        print(line, end="")
if p.wait():
    raise RuntimeError(f"run_all.py failed (exit {p.returncode}); re-run the notebook to resume from Drive")

In [ ]:
# Step 7: Results
print(open(os.path.join(os.environ["SSV_RESULTS"], "RESULTS.md")).read())

In [ ]:
# Step 8: Flush Drive & unassign GPU
auto_teardown()